# Notebook 4: Long-term Displacement and Return Analysis

This notebook analyzes weekly return rates of evacuees and runs logistic regressions predicting long-term displacement outcomes based on social homophily of evacuation destinations.

**Paper reference:** Figure 4 — Weekly return rates and displacement regression; Supplementary Tables S21–S28.

**Inputs:** `evacuees_for_regression_final.csv`, `displaced_for_regression_final.csv`, `colorado_SCI_ZIP_CBG.csv`

**Outputs:** Return rate curves, displacement regression results


In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler

In [ ]:
SCI = pd.read_csv("colorado_SCI_ZIP_CBG.csv")

In [ ]:
# Check for duplicates in SCI based on 'cbg_user' and 'cbg_fr'
duplicates = SCI.duplicated(subset=['cbg_user', 'cbg_fr'], keep=False)
if duplicates.sum() > 0:
    print(f"There are {duplicates.sum()} duplicate rows in SCI based on 'cbg_user' and 'cbg_fr'.")

# If duplicates exist, you might want to aggregate or remove them.
# For example, you can take the mean of 'scaled_sci' for each combination:
SCI_unique = SCI.groupby(['cbg_user', 'cbg_fr'], as_index=False)['scaled_sci'].mean()

# Aggregate the duplicates by taking the mean of 'scaled_sci' for each unique pair of 'cbg_user' and 'cbg_fr'
SCI_unique = SCI.groupby(['cbg_user', 'cbg_fr'], as_index=False)['scaled_sci'].mean()

# Check the shape of the resulting dataframe to ensure duplicates are removed
print(f"Shape of SCI after removing duplicates: {SCI_unique.shape}")

In [ ]:
SCI_unique.head()

In [ ]:
alt_disp = pd.read_csv("displaced_for_regression_final.csv")
alt_disp.columns

In [ ]:
Selct_cols = ["Unnamed: 0","pre_crisis_home_cbg","last_known_post_crisis_home_cbg",
             'Or_population_density', 'Or_unemployment_rate', 'radius_of_gyration',
       'Or_white_population', 'Or_black_population',
       'Or_asian_population', 'avg_entropy','homophily' ]
alt_disp_subset = alt_disp[Selct_cols]
alt_disp_subset.rename(columns={'homophily': 'disp_homophily'}, inplace=True)
alt_disp_subset.head()

In [ ]:
evacuees = pd.read_csv("evacuees_for_regression_final.csv")

In [ ]:
evacuees.head()

In [ ]:
evacuees.columns


In [ ]:
displaced = pd.merge(
    evacuees, 
    alt_disp_subset, 
    left_on=['Unnamed: 0'],
    right_on=['Unnamed: 0'], 
    how='left' 
    )

In [ ]:
# Check how many rows in 'last_known_post_crisis_home_cbg' are NaN
nan_count = displaced['last_known_post_crisis_home_cbg'].isna().sum()
print(f"Number of NaN values in 'last_known_post_crisis_home_cbg': {nan_count}")

# Fill NaN values in 'last_known_post_crisis_home_cbg' with values from 'crisis_home_cbg'
displaced['last_known_post_crisis_home_cbg'].fillna(displaced['crisis_home_cbg'], inplace=True)


In [ ]:
# Check how many rows in 'last_known_post_crisis_home_cbg' are NaN
nan_count = displaced['disp_homophily'].isna().sum()
print(f"Number of NaN values in 'last_known_post_crisis_home_cbg': {nan_count}")

# Fill NaN values in 'last_known_post_crisis_home_cbg' with values from 'crisis_home_cbg'
displaced['disp_homophily'].fillna(displaced['homophily'], inplace=True)


In [ ]:
displaced.shape

In [ ]:
displaced.head()

In [ ]:
displaced.shape

In [ ]:
displaced_newsci = pd.merge(
    displaced, 
    SCI_unique, 
    left_on=['pre_crisis_home_cbg_x', 'last_known_post_crisis_home_cbg'],
    right_on=['cbg_user', 'cbg_fr'], 
    how='left' 
    )

In [ ]:
displaced_newsci.rename(columns={'scaled_sci': 'disp_scaled_sci'}, inplace=True)
displaced_newsci

In [ ]:
displaced_bothsci = pd.merge(
    displaced_newsci, 
    SCI_unique, 
    left_on=['pre_crisis_home_cbg_x', 'crisis_home_cbg'],
    right_on=['cbg_user', 'cbg_fr'], 
    how='left' 
    )
displaced_bothsci.columns

In [ ]:
#Selct_cols = ["Unnamed: 0","pre_crisis_home_cbg_x",'crisis_home_cbg', "last_known_post_crisis_home_cbg", 'motif_final', 'last_known_post_crisis_home_cbg', 'homophily', 'scaled_sci', 'D1_impact' ]
#displaced_newsci = displaced_newsci[Selct_cols]
displaced_bothsci.head()

In [ ]:
displaced_bothsci.to_csv("evac_n_disp.csv.gz")

In [ ]:
evacuees_newSCI = pd.read_csv("evac_n_disp.csv.gz")

# for line graph 
Similarity and connectedness ~ percentage returned

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
# Create new columns indicating if returned in the first month or by six months
displaced_newsci['returned_1_month'] = displaced_newsci['motif_final'].apply(lambda x: 1 if x == 'Evacuated_and_Returned_Immediately' else 0)
displaced_newsci['returned_6_months'] = displaced_newsci['motif_final'].apply(lambda x: 1 if x in ['Evacuated_and_Returned_Immediately', 'Evacuated_and_Returned_Later'] else 0)
displaced_newsci['displaced'] = displaced_newsci['motif_final'].apply(lambda x: 1 if x in ['Evacuated_and_displaced', 'Evacuated_never_Returned'] else 0)

# Ffitin logistic regression to plot relationship for both time periods and displaces
def plot_logistic_regression_relationship(x_column, xlabel, ylabel, title, color):
    # Drop rows with NaN in the relevant columns
    filtered_data = displaced_newsci[[x_column, 'returned_1_month', 'returned_6_months', 'displaced']].dropna()

    X = filtered_data[[x_column]].values.reshape(-1, 1)

    # Logistic Regression for the first month return
    model_1_month = LogisticRegression().fit(X, filtered_data['returned_1_month'])
    X_range = np.linspace(X.min(), X.max(), 500).reshape(-1, 1)  # Generate smooth values
    y_pred_1_month = model_1_month.predict_proba(X_range)[:, 1] * 100  # Convert to percentages

    # Logistic Regression for the six-month return
    model_6_months = LogisticRegression().fit(X, filtered_data['returned_6_months'])
    y_pred_6_months = model_6_months.predict_proba(X_range)[:, 1] * 100  # Convert to percentages

    # Logistic Regression for displced
    model_displaced = LogisticRegression().fit(X, filtered_data['displaced'])
    y_pred_displaced = model_displaced.predict_proba(X_range)[:, 1] * 100  # Convert to percentages

    # Plot the logistic regression lines
    plt.figure(figsize=(4, 3))
    
    # Plot the first-month return line
    plt.plot(X_range, y_pred_1_month, label='Returned in 1 Month', color=color, linestyle='-')
    
    # Plot the six-month return line
    plt.plot(X_range, y_pred_6_months, label='Returned by 6 Months', color=color, linestyle='--')

    # Plot the displaced line
    plt.plot(X_range, y_pred_displaced, label='Displaced', color=color, linestyle=':')

    # Customize plot
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    #plt.legend()
    plt.tight_layout()
    plt.show()

plot_logistic_regression_relationship(
    x_column='scaled_sci', 
    xlabel='Familiarity (D2_scaled_sci)', 
    ylabel='Percentage Returned', 
    title='Familiarity vs. Percentage Return', 
    color='chocolate'
)

plot_logistic_regression_relationship(
    x_column='disp_homophily', 
    xlabel='Similarity (Homophily)', 
    ylabel='Percentage Returned', 
    title='Similarity vs. Percentage Return', 
    color='steelblue'
)

In [ ]:
displaced_newsci.columns

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Step 1: Create bins for familiarity (scaled_sci) and similarity (disp_homophily)
displaced_newsci['familiarity_bin'] = pd.cut(displaced_newsci['scaled_sci'], bins=5, labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])
displaced_newsci['similarity_bin'] = pd.cut(displaced_newsci['disp_homophily'], bins=5, labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])

# Step 2: Calculate percentages for each bin for Familiarity
familiarity_grouped = displaced_newsci.groupby('familiarity_bin').agg(
    total_count=('Unnamed: 0', 'count'),  # Total individuals in the bin
    returned_1_month_sum=('returned_1_month', 'sum'),  # Total returned in 1 month
    returned_6_months_sum=('returned_6_months', 'sum'),  # Total returned in 6 months
    displaced_sum=('displaced', 'sum')  # Total displaced
)

# Subtract returned_1_month_sum from returned_6_months_sum to get those who returned only after 1 month but within 6 months
familiarity_grouped['returned_only_in_6_months'] = familiarity_grouped['returned_6_months_sum'] - familiarity_grouped['returned_1_month_sum']

# Calculate the percentage of each category within each bin
familiarity_grouped['returned_1_month_pct'] = (familiarity_grouped['returned_1_month_sum'] / familiarity_grouped['total_count']) * 100
familiarity_grouped['returned_6_months_pct'] = (familiarity_grouped['returned_only_in_6_months'] / familiarity_grouped['total_count']) * 100
familiarity_grouped['displaced_pct'] = (familiarity_grouped['displaced_sum'] / familiarity_grouped['total_count']) * 100

# Step 3: Plot stacked bar chart for Familiarity with distinct shades from "Oranges" color map
color_palette_fam = [plt.get_cmap('Oranges')(0.3), plt.get_cmap('Oranges')(0.6), plt.get_cmap('Oranges')(0.9)]

familiarity_grouped[['returned_1_month_pct', 'returned_6_months_pct', 'displaced_pct']].plot(
    kind='bar', stacked=True, figsize=(8, 5), color=color_palette_fam
)

plt.xlabel('Familiarity (D2_scaled_sci) Bins')
plt.ylabel('Percentage of Individuals (%)')
# Explicitly remove the legend
plt.legend().set_visible(False)
# Do not call plt.title() to remove the title
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Step 4: Calculate percentages for each bin for Similarity
similarity_grouped = displaced_newsci.groupby('similarity_bin').agg(
    total_count=('Unnamed: 0', 'count'),  # Total individuals in the bin
    returned_1_month_sum=('returned_1_month', 'sum'),  # Total returned in 1 month
    returned_6_months_sum=('returned_6_months', 'sum'),  # Total returned in 6 months
    displaced_sum=('displaced', 'sum')  # Total displaced
)

# Subtract returned_1_month_sum from returned_6_months_sum to get those who returned only after 1 month but within 6 months
similarity_grouped['returned_only_in_6_months'] = similarity_grouped['returned_6_months_sum'] - similarity_grouped['returned_1_month_sum']

# Calculate the percentage of each category within each bin
similarity_grouped['returned_1_month_pct'] = (similarity_grouped['returned_1_month_sum'] / similarity_grouped['total_count']) * 100
similarity_grouped['returned_6_months_pct'] = (similarity_grouped['returned_only_in_6_months'] / similarity_grouped['total_count']) * 100
similarity_grouped['displaced_pct'] = (similarity_grouped['displaced_sum'] / similarity_grouped['total_count']) * 100

# Step 5: Plot stacked bar chart for Similarity with distinct shades from "Blues" color map
color_palette_sim = [plt.get_cmap('Blues')(0.3), plt.get_cmap('Blues')(0.6), plt.get_cmap('Blues')(0.9)]

similarity_grouped[['returned_1_month_pct', 'returned_6_months_pct', 'displaced_pct']].plot(
    kind='bar', stacked=True, figsize=(8, 5), color=color_palette_sim
)

plt.xlabel('Similarity (Homophily) Bins')
plt.ylabel('Percentage of Individuals (%)')
# Explicitly remove the legend
plt.legend().set_visible(False)
# Do not call plt.title() to remove the title
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Step 1: Create bins for familiarity (scaled_sci) and similarity (disp_homophily)
displaced_newsci['familiarity_bin'] = pd.cut(displaced_newsci['scaled_sci'], bins=5, labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])
displaced_newsci['similarity_bin'] = pd.cut(displaced_newsci['disp_homophily'], bins=5, labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])

# Step 2: Calculate percentages for each bin for Familiarity
familiarity_grouped = displaced_newsci.groupby('familiarity_bin').agg(
    total_count=('Unnamed: 0', 'count'),  # Total individuals in the bin
    returned_1_month_sum=('returned_1_month', 'sum'),  # Total returned in 1 month
    returned_6_months_sum=('returned_6_months', 'sum'),  # Total returned in 6 months
    displaced_sum=('displaced', 'sum')  # Total displaced
)

# Subtract returned_1_month_sum from returned_6_months_sum to get those who returned only after 1 month but within 6 months
familiarity_grouped['returned_only_in_6_months'] = familiarity_grouped['returned_6_months_sum'] - familiarity_grouped['returned_1_month_sum']

# Calculate the percentage of each category within each bin
familiarity_grouped['returned_1_month_pct'] = (familiarity_grouped['returned_1_month_sum'] / familiarity_grouped['total_count']) * 100
familiarity_grouped['returned_6_months_pct'] = (familiarity_grouped['returned_only_in_6_months'] / familiarity_grouped['total_count']) * 100
familiarity_grouped['displaced_pct'] = (familiarity_grouped['displaced_sum'] / familiarity_grouped['total_count']) * 100

# Step 3: Plot stacked bar chart for Familiarity with distinct shades from "Oranges" color map
color_palette_fam = [plt.get_cmap('Oranges')(0.3), plt.get_cmap('Oranges')(0.6), plt.get_cmap('Oranges')(0.9)]

ax = familiarity_grouped[['returned_1_month_pct', 'returned_6_months_pct', 'displaced_pct']].plot(
    kind='bar', stacked=True, figsize=(8, 5), color=color_palette_fam
)

# Add percentage markers on the bars
for p in ax.patches:
    width = p.get_width()
    height = p.get_height()
    x, y = p.get_xy() 
    if height > 0:
        ax.text(x + width / 2, y + height / 2, f'{height:.1f}%', ha='center', va='center', fontsize=15, color='white')

plt.xlabel('Familiarity (D2_scaled_sci) Bins')
plt.ylabel('Percentage of Individuals (%)')
# Explicitly remove the legend
plt.legend().set_visible(False)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Step 4: Calculate percentages for each bin for Similarity
similarity_grouped = displaced_newsci.groupby('similarity_bin').agg(
    total_count=('Unnamed: 0', 'count'),  # Total individuals in the bin
    returned_1_month_sum=('returned_1_month', 'sum'),  # Total returned in 1 month
    returned_6_months_sum=('returned_6_months', 'sum'),  # Total returned in 6 months
    displaced_sum=('displaced', 'sum')  # Total displaced
)

# Subtract returned_1_month_sum from returned_6_months_sum to get those who returned only after 1 month but within 6 months
similarity_grouped['returned_only_in_6_months'] = similarity_grouped['returned_6_months_sum'] - similarity_grouped['returned_1_month_sum']

# Calculate the percentage of each category within each bin
similarity_grouped['returned_1_month_pct'] = (similarity_grouped['returned_1_month_sum'] / similarity_grouped['total_count']) * 100
similarity_grouped['returned_6_months_pct'] = (similarity_grouped['returned_only_in_6_months'] / similarity_grouped['total_count']) * 100
similarity_grouped['displaced_pct'] = (similarity_grouped['displaced_sum'] / similarity_grouped['total_count']) * 100

# Step 5: Plot stacked bar chart for Similarity with distinct shades from "Blues" color map
color_palette_sim = [plt.get_cmap('Blues')(0.3), plt.get_cmap('Blues')(0.6), plt.get_cmap('Blues')(0.9)]

ax = similarity_grouped[['returned_1_month_pct', 'returned_6_months_pct', 'displaced_pct']].plot(
    kind='bar', stacked=True, figsize=(8, 5), color=color_palette_sim
)

# Add percentage markers on the bars
for p in ax.patches:
    width = p.get_width()
    height = p.get_height()
    x, y = p.get_xy() 
    if height > 0:
        ax.text(x + width / 2, y + height / 2, f'{height:.1f}%', ha='center', va='center', fontsize=15, color='white')

plt.xlabel('Similarity (Homophily) Bins')
plt.ylabel('Percentage of Individuals (%)')
# Explicitly remove the legend
plt.legend().set_visible(False)
#plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# for regression

In [ ]:
motif_mapping = {
    'Evacuated_and_displaced': 0,
    'Evacuated_never_Returned': 0,
    'Evacuated_and_Returned_Immediately': 1,
    'Evacuated_and_Returned_Later': 1
}

evacuees_newSCI['displacement_status'] = evacuees_newSCI['motif_final'].map(motif_mapping)


In [ ]:
evacuees_newSCI.head()

In [ ]:
# Drop rows where 'scaled_sci' is NaN
evacuees_newSCI_cleaned = evacuees_newSCI.dropna(subset=['scaled_sci'])

# Optionally, check the shape of the dataframe to see how many rows are left
print(f"Number of rows after dropping missing 'scaled_sci': {evacuees_newSCI_cleaned.shape[0]}")

In [ ]:
evacuees_newSCI_cleaned['displacement_status'].value_counts()

In [ ]:
evacuees_newSCI_cleaned.head()

In [ ]:
missing_SCI_count = evacuees_newSCI_cleaned['scaled_sci'].isna().sum()
print(missing_SCI_count)

In [ ]:
df = evacuees_newSCI_cleaned.copy()

df.loc[:, 'log_scaled_sci'] = df['scaled_sci'].apply(np.log1p)

independent_vars = [
    'homophily',
    'log_scaled_sci'
]

y = df['displacement_status']

df_clean = df.dropna(subset=['displacement_status', 'scaled_sci'])

X = df_clean[independent_vars]
y = df_clean['displacement_status']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled = pd.DataFrame(X_scaled, columns=independent_vars, index=X.index)

X_scaled = sm.add_constant(X_scaled)

X_scaled = X_scaled.loc[y.index]

model = sm.Logit(y, X_scaled)
result = model.fit()

print(result.summary())

In [ ]:
# Function to plot significant coefficients with error bars
def plot_significant_coefficients(model, X_columns, significance_level=0.07):
    # Extract coefficients, standard errors, and p-values from the model
    coef_df = pd.DataFrame({
        'Variable': model.params.index[1:],  # Extract variable names from the model
        'Coefficient': model.params[1:],  # Exclude the intercept
        'Error': model.bse[1:],  # Exclude the intercept
        'PValue': model.pvalues[1:]  # Exclude the intercept
    })

    # Filter to keep only significant coefficients
    significant_coef_df = coef_df[coef_df['PValue'] < significance_level]

    # Rename variable labels to more descriptive names
    variable_labels = {
        'log_Or_population_density': 'Population\nDensity',
        'log_Or_unemployment_rate': 'Unemployment',
        'Or_total_housing_units_r': 'Total\nHousing Units',
        'log_Or_white_population': 'White\nPopulation',
        'log_black_population': 'Black\nPopulation',
        'log_Or_asian_population': 'Asian\nPopulation',
        'Or_median_household_income': 'Median\nIncome',
        'Or_education_atleast_one_degree': 'Education',
        'log_homophily' : 'Similarity',
        'log_scaled_sci': 'Familiarity',
        'radius_of_gyration': "Radius of\nGyration", 
        'avg_entropy': 'Entropy', 
        'dist_epicentre': 'Distance from Fire',
        'log_distance_OD': 'Distance\nOrigin and Destination'
    }

    significant_coef_df['Variable'] = significant_coef_df['Variable'].map(variable_labels).fillna(significant_coef_df['Variable'])

    # Plotting the significant coefficients with error bars
    plt.figure(figsize=(4, 3))
    plt.errorbar(x=significant_coef_df['Coefficient'], y=significant_coef_df['Variable'],
                 xerr=significant_coef_df['Error'], fmt='o', ecolor='darkgrey', 
                 capsize=5, markersize=8, linestyle='None', markerfacecolor='none', 
                 markeredgewidth=1, color='teal')

    # Customize the spines (borders) and set light grey lines for x and y axes
    ax = plt.gca()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Set the left and bottom spines (y and x axes) to light grey
    ax.spines['left'].set_color('lightgrey')
    ax.spines['bottom'].set_color('lightgrey')
    
    # Set the ticks and tick labels color to light grey
    ax.tick_params(axis='x', colors='black')
    ax.tick_params(axis='y', colors='black')

    # Add vertical line at 0 to indicate no effect
    plt.axvline(x=0, color='lightgrey', linewidth=0.8, linestyle='--')

    # Customize the plot appearance
    plt.xlabel('Beta Coefficients')
    plt.ylabel('Variables')
    #plt.title('Significant Coefficients with 95% CI')
    plt.tight_layout()
    # Save the plot

    plt.savefig(f"coefficients_plot.png", dpi=300, bbox_inches='tight')
    plt.show()
    

# Plot coefficients for the logistic regression model
plot_significant_coefficients(result, X_scaled.columns)

In [ ]:
df.groupby('displacement_status')[independent_vars].median()

In [ ]:
df.to_csv("evacuees_newSCI_cleaned.csv")